# HeartTwin: KIT and EDGAR real-data benchmarks
Pinned source: `351aaa776f414087afb0fd49575080805e34ff81`.
Runs on a standard CPU Colab runtime. Raw downloads come only from the dataset owners, with pinned checksums. No hospital data or credentials are used.

KIT: original anatomy/data audit plus an explicitly assumed native 0D circulation fit (23 calibration samples, 12 temporal holdout samples; RV end-systolic volume is a separate chamber target). The split is retrospective after raw-data inspection; this is not preregistered or prospectively blinded. No selection or retuning on holdout.

EDGAR Bratislava P034: audit the actual documentation, surface geometry and electrode mapping. At inspection, the signed-in source page linked no Run1/Run2 waveform archive. Electrical prediction metrics must remain absent unless those missing files are obtained and qualified.

KIT data licence: CC BY-NC 4.0, Gerach et al.; see source receipts. EDGAR attribution and rights are retained in its receipts. This is not clinical validation or a complete spatial whole-heart benchmark.



In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, traceback, zipfile
SOURCE_COMMIT = '351aaa776f414087afb0fd49575080805e34ff81'
CARDIMECH_COMMIT = '5dc14d886ccbda5a1398b863f9ae7821bbffa18b'
ROOT = Path('/content/HeartTwin_KIT_EDGAR_20261020/retained-' + SOURCE_COMMIT[:12])
ROOT.mkdir(parents=True, exist_ok=True)
DATA = ROOT.parent
RESULTS = ROOT/'results'; RESULTS.mkdir(exist_ok=True)
ENV = dict(os.environ, MPLBACKEND='Agg', PYTHONUNBUFFERED='1')
def run(command, name, timeout=900):
    print('Starting', name, flush=True)
    with (RESULTS/(name+'.log')).open('w') as log:
        completed = subprocess.run(list(map(str, command)), stdout=log, stderr=subprocess.STDOUT, env=ENV, timeout=timeout)
    if completed.returncode:
        print((RESULTS/(name+'.log')).read_text()[-10000:])
        raise RuntimeError(name+' failed: '+str(completed.returncode))
repo = ROOT/'source'
if not repo.exists():
    run(['git','init',repo], 'git-init')
    run(['git','-C',repo,'fetch','--depth','1','https://github.com/Virelion-Biotech/Virelion-HeartTwin.git',SOURCE_COMMIT], 'git-fetch')
    run(['git','-C',repo,'checkout','--detach','FETCH_HEAD'], 'git-checkout')
assert subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()==SOURCE_COMMIT
assert not subprocess.check_output(['git','-C',str(repo),'status','--porcelain','--untracked-files=no'],text=True).strip()
venv = ROOT/'venv'
if not venv.exists(): run([sys.executable,'-m','venv','--without-pip',venv], 'venv')
python = venv/'bin/python'
ENV['PATH'] = str(python.parent)+os.pathsep+ENV['PATH']
run([sys.executable,'-m','pip','--python',python,'install','pip==25.1.1','numpy==2.2.6','scipy==1.16.3','pydantic==2.12.3','PyYAML==6.0.3','meshio==5.3.5','matplotlib==3.10.7','ruff==0.14.1'], 'install-base')
run([python,'-m','pip','install','-e',repo,'git+https://github.com/Virelion-Biotech/Virelion-CardiMech.git@'+CARDIMECH_COMMIT], 'install-projects')
run([python,'-m','pip','check'], 'pip-check')
(RESULTS/'source_pin.json').write_text(json.dumps({'source_commit':SOURCE_COMMIT,'CardiMech_commit':CARDIMECH_COMMIT,'original_KIT_run_commit':'492a83a2a3b4f9627892f77ff7f958a72ffe511e','execution':'User Edge Colab','scope':'Exploratory KIT circulation plus EDGAR geometry audit; no whole-heart accuracy claim'},indent=2))
state={'status':'STARTED','KIT':'NOT_RUN','EDGAR':'NOT_RUN','clinical_qualification':'NOT_ESTABLISHED'}
try:
    run([python,repo/'scripts/benchmark_kit_edgar.py','--data-dir',DATA,'--output',RESULTS], 'KIT-benchmark', timeout=600)
    state['KIT']='COMPLETED_EXPLORATORY'
    run([python,repo/'scripts/benchmark_edgar_p034.py','--data-dir',DATA/'edgar','--output',RESULTS], 'EDGAR-audit', timeout=300)
    state['EDGAR']=json.loads((RESULTS/'EDGAR_status.json').read_text())['status']
    run([python,'-m','py_compile',repo/'scripts/benchmark_kit_edgar.py',repo/'scripts/benchmark_edgar_p034.py'], 'compile')
    run([python,'-m','ruff','check',repo/'scripts/benchmark_kit_edgar.py',repo/'scripts/benchmark_edgar_p034.py'], 'lint')
    state['status']='EXPLORATORY_RUN_COMPLETE'
except BaseException:
    state.update(status='FAILED',error=traceback.format_exc())
    print(state['error'])
finally:
    (RESULTS/'RUN_STATUS.json').write_text(json.dumps(state,indent=2))
print(json.dumps(state,indent=2))
if state['status']=='FAILED': raise RuntimeError('Retain failure logs; do not report a pass.')

